# DeepGate-FaceRec: Metric Learning & Triplet Training

In this notebook, you will train **`FaceEmbeddingNet`** using **online semi-hard triplet mining** so that face images of the same person map close together on a 128-dimensional unit hypersphere ($\Vert{}z\Vert{}_2 = 1$), while images of different people are pushed apart by at least a margin $\alpha$.

## How Triplet Metric Learning Works

1. **PK Batch Sampling:** Each mini-batch samples $P$ distinct identities with $K$ images per identity ($B = P \times K$).
2. **Feature Extraction:** All images pass through the **same** CNN backbone (`ResNet-18`) and $L_2$-normalized projection head (`FaceEmbeddingNet`).
3. **Online Semi-Hard Mining & Loss:** Inside the batch, we compute pairwise squared Euclidean distances $D(z_i, z_j) = \Vert{}z_i - z_j\Vert{}_2^2$ and select negatives satisfying $D(z_a, z_p) < D(z_a, z_n) < D(z_a, z_p) + \alpha$, minimizing:
   $$\mathcal{L}_{\text{triplet}}(a, p, n) = \max\Big(0,\; \Vert{}z_a - z_p\Vert{}_2^2 - \Vert{}z_a - z_n\Vert{}_2^2 + \alpha\Big)$$

![training_process](assets/model_training.jpeg)


---

### Tracking Training History (`EpochStats`)

`MetricTrainer.fit()` returns a `list[EpochStats]` recording both optimization dynamics and 1:1 verification metrics on unseen validation identities after each epoch:

```python
@dataclass
class EpochStats:
    """Container for per-epoch training and validation metrics.

    Attributes:
        train_loss: Average triplet margin loss across processed training batches.
        active_triplets_ratio: Fraction in ``[0.0, 1.0]`` of mined triplets that violated
            the margin (``d(a, p) - d(a, n) + margin > 0``).
        val_accuracy: Best 1:1 verification accuracy on validation pairs.
        val_auc: ROC Area Under the Curve on validation pairs.
        val_eer: Equal Error Rate on validation pairs (lower is better).
        val_margin: Separation gap ``mean_neg_dist - mean_pos_dist`` on validation pairs.
        epoch: 1-based epoch index.
        is_best: ``True`` if this epoch achieved the best monitored validation score so far.
    """

    train_loss: float
    active_triplets_ratio: float
    val_accuracy: float
    val_auc: float
    val_eer: float
    val_margin: float
    epoch: int = 0
    is_best: bool = False

```

---

### Report Deliverables Produced in This Notebook

Make sure to generate and include the following two figures in your technical report:

1. **Training & Validation Learning Curves:** Plot `train_loss`, `active_triplets_ratio`, `val_accuracy`, `val_auc`, `val_eer`, and `val_margin` across epochs to analyze convergence, overfitting, and whether active margin violations decrease smoothly as the separation gap widens.
2. **Before vs. After Embedding Space Visualization:** Compare 2D projections (`t-SNE` or `UMAP`) and **Cosine Silhouette scores** of unseen validation identities extracted from an untrained pretrained backbone (`Before`) against your fine-tuned triplet model (`After`) to demonstrate generalization to novel people.

```



In [1]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
# Your imports go here.

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# Project-relative paths (adjust if your notebook sits inside a notebooks/ subfolder)
DATA_ROOT = Path("data/faces")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
CHECKPOINT_DIR = None # Fill me please

Using device: mps
